# Apache Beam Lab 3: Core Transforms

## Overview
This lab focuses on core Apache Beam transforms that form the building blocks of data processing pipelines. Mastering these transforms is essential for building effective data pipelines.

## Learning Objectives
- Master ParDo transform for custom processing
- Learn advanced Map and FlatMap patterns
- Understand GroupByKey and CoGroupByKey
- Practice Combine transforms for aggregation
- Build custom DoFn classes

## Prerequisites
- Completed Lab 2: Pipeline Fundamentals
- Understanding of basic Python functions

## ParDo Transform
ParDo (Parallel Do) is the most fundamental transform in Apache Beam. It applies a function to each element in a PCollection.

In [ ]:
import apache_beam as beam

# Basic ParDo example
class UppercaseFn(beam.DoFn):
    def process(self, element):
        yield element.upper()

def basic_pardo():
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create words' >> beam.Create(['hello', 'world', 'beam'])
            | 'Uppercase' >> beam.ParDo(UppercaseFn())
            | 'Print' >> beam.Map(print)
        )

print("ParDo concept: Apply custom processing to each element")

## Exercise 1: Custom Data Processing with ParDo
Create a custom DoFn that processes sales data and enriches it with additional calculations.

In [ ]:
# Your code here to create a custom DoFn for sales data enrichment
# 1. Create a DoFn that calculates total price and applies discount rules
# 2. Process sample sales data
# 3. Output enriched data

import pandas as pd
import os

class EnrichSalesFn(beam.DoFn):
    def process(self, element):
        # Calculate total price
        total = element['quantity'] * element['price']
        
        # Apply discount rules
        discount = 0.0
        if total > 1000:
            discount = 0.10  # 10% discount for large orders
        elif total > 500:
            discount = 0.05  # 5% discount for medium orders
        
        # Enrich the record
        enriched = {
            **element,
            'total_price': total,
            'discount_percentage': discount,
            'discounted_total': total * (1 - discount),
            'discount_amount': total * discount
        }
        yield enriched

def enrich_sales_data():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create sales data' >> beam.Create(data)
            | 'Enrich data' >> beam.ParDo(EnrichSalesFn())
            | 'Format output' >> beam.Map(
                lambda x: f"Transaction {x['transaction_id']}: ${x['discounted_total']:.2f} (saved ${x['discount_amount']:.2f})")
            | 'Print' >> beam.Map(print)
        )

print("Enriching sales data with custom ParDo...")
enrich_sales_data()

## GroupByKey Transform
GroupByKey groups values by their keys. This is essential for many data processing patterns.

In [ ]:
def group_by_key_example():
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create pairs' >> beam.Create([
                ('apple', 1), ('banana', 2), ('apple', 3), 
                ('orange', 4), ('banana', 5), ('apple', 6)
            ])
            | 'Group by key' >> beam.GroupByKey()
            | 'Format' >> beam.Map(lambda x: f"{x[0]}: {list(x[1])}")
            | 'Print' >> beam.Map(print)
        )

print("GroupByKey concept: Group values by their keys")

## Exercise 2: Sales Aggregation with GroupByKey
Use GroupByKey to aggregate sales data by different dimensions.

In [ ]:
# Your code here to aggregate sales data using GroupByKey
# 1. Group sales by product
# 2. Group sales by customer
# 3. Calculate statistics for each group

def aggregate_sales_by_key():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        # Aggregate by product
        (
            pipeline
            | 'Create data' >> beam.Create(data)
            | 'Extract product total' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Group by product' >> beam.GroupByKey()
            | 'Sum values' >> beam.Map(lambda x: (x[0], sum(x[1])))
            | 'Format' >> beam.Map(lambda x: f"Product {x[0]}: ${x[1]:.2f}")
            | 'Print' >> beam.Map(print)
        )

print("Aggregating sales data with GroupByKey...")
aggregate_sales_by_key()

## Combine Transforms
Combine transforms are more efficient than GroupByKey for simple aggregations.

In [ ]:
def combine_example():
    with beam.Pipeline() as pipeline:
        # CombinePerKey - more efficient than GroupByKey + sum
        (
            pipeline
            | 'Create pairs' >> beam.Create([
                ('a', 1), ('b', 2), ('a', 3), ('b', 4), ('a', 5)
            ])
            | 'Combine per key' >> beam.CombinePerKey(sum)
            | 'Print' >> beam.Map(print)
        )
        
        # CombineGlobally - aggregate all elements
        (
            pipeline
            | 'Create numbers' >> beam.Create([1, 2, 3, 4, 5])
            | 'Combine globally' >> beam.CombineGlobally(sum)
            | 'Print total' >> beam.Map(lambda x: print(f"Total: {x}"))
        )

print("Combine concept: Efficient aggregation operations")

## Exercise 3: Advanced Aggregation
Create a pipeline that performs multiple aggregations using Combine transforms.

In [ ]:
# Your code here to perform advanced aggregations
# 1. Calculate total revenue per product
# 2. Find average transaction value
# 3. Count transactions per customer
# Use Combine transforms for efficiency

def advanced_aggregation():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        sales = pipeline | 'Create data' >> beam.Create(data)
        
        # Total revenue per product
        (
            sales
            | 'Extract product revenue' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Combine revenue' >> beam.CombinePerKey(sum)
            | 'Format revenue' >> beam.Map(
                lambda x: f"Product {x[0]}: ${x[1]:.2f}")
            | 'Print revenue' >> beam.Map(print)
        )
        
        # Average transaction value
        (
            sales
            | 'Calculate transaction totals' >> beam.Map(
                lambda x: x['quantity'] * x['price'])
            | 'Calculate average' >> beam.CombineGlobally(
                lambda values: sum(values) / len(values) if values else 0)
            | 'Print average' >> beam.Map(
                lambda x: print(f"Average transaction: ${x:.2f}"))
        )
        
        # Transaction count per customer
        (
            sales
            | 'Extract customer' >> beam.Map(lambda x: (x['customer_id'], 1))
            | 'Count transactions' >> beam.CombinePerKey(sum)
            | 'Format counts' >> beam.Map(
                lambda x: f"Customer {x[0]}: {x[1]} transactions")
            | 'Print counts' >> beam.Map(print)
        )

print("Performing advanced aggregations...")
advanced_aggregation()

## Summary

In this lab, you have:
- Mastered ParDo transform for custom processing
- Built custom DoFn classes
- Learned GroupByKey for grouping operations
- Used Combine transforms for efficient aggregation
- Applied these transforms to real data processing scenarios

## Key Takeaways
- ParDo is the most flexible transform for custom logic
- GroupByKey is powerful but can be less efficient than Combine
- CombinePerKey is optimized for simple aggregations
- Custom DoFn classes enable complex processing logic
- Choosing the right transform impacts pipeline performance

## Next Steps

Proceed to [Lab 4: I/O Connectors](lab-04-io-connectors.md) to learn about reading and writing data from various sources.